# Challenge 1 — Climate-Resilient Walkability

### Load & Explore

Welcome. By the end of this notebook you will have four tables in your own BigQuery project,
you will understand where every row came from, and you will know exactly what your team is
building for the rest of the day.

**Read the text, not just the code.** This notebook is written so that you can understand
what is happening by reading only the narrative between the cells. If you have never used
BigQuery or Google Cloud before, you will still be able to follow along. If you have, you
can skim the explanations and run the cells.

---

## What your team is building today

An agent that tells a city planner **where to put a cooling center, a hydration truck, a
warming shelter, or a stand of new trees**—not "somewhere downtown," but a specific location,
with the reasoning attached.

The hard part is the word *microclimate*. Two blocks in the same city, at the same hour on the
same day, can differ by more than fifteen degrees. One has tree cover and low buildings. The
other is parking lot. A planner deciding where a cooling center goes needs to know which is
which, and today you are going to build the thing that tells them.

## What this notebook does

It gets you data. That is all.

It pulls from four public sources, cleans them, explains every cleaning decision, and loads
the results into BigQuery in **your** project. It does **not** build your model and it does
**not** build your agent. Those are yours.

## What this notebook does not do

There is no `CREATE MODEL` statement here and no agent code. Your required differentiating
technology is **BigQuery ML**, and the model is the part that is genuinely yours to design.
Section 9 sets up the problem and tells you what the free tier covers. It stops there
deliberately.

## Where this fits in your day

Your table is eight to ten people, which is too many to have everyone typing into one file.
The work splits into four lanes that run in parallel:

| Lane | What they do | Starts |
|---|---|---|
| **Data** | This notebook, then the BQML model | now |
| **Agent** | ADK agent, tools, the MCP server | now |
| **Front end** | The face: custom UI, Gemini Enterprise, or `adk web` | now, against a mock |
| **Story** | The Placement Recommendation, the demo narrative | now |

If you are reading this, you are probably the data lane. **Budget about 40 minutes.** The
single most useful thing you can do for your team is finish this and start the model early,
because the agent lane cannot call a prediction that does not exist yet.

---

# 1. Setup

## What we are about to do

Three things: work out which Google Cloud project we are in, pick a place to put our tables,
and create it.

**If you have used BigQuery before:** we are creating one dataset in the US multi-region and
setting a few constants. Skim the code, change `CITY` to whatever you like, and move on.

**If you have never used Google Cloud before**, here is the vocabulary, because it is genuinely
confusing the first time:

- A **project** is your workspace. It is where everything you create today lives, and it is
  already made for you—you do not need to create one.
- **BigQuery** is a database that holds enormous tables and answers questions about them in
  seconds. You talk to it in SQL.
- A **dataset** in BigQuery is a folder. Tables go inside it. We are making one folder called
  `a4i_climate` and everything today goes in there.
- A **location** is which part of the world your data physically sits in. We use `US`. The one
  rule worth remembering is that things you join together generally need to be in the same
  location, so pick one and stay with it.

You are running in **Colab Enterprise**, which means this notebook is already signed in as you,
in your project. There is no authentication step and no key file to download. That is why we
are using it.

In [ ]:
# --- Configuration -----------------------------------------------------------
# The only line most teams will change is CITY.

CITY = "Chicago"          # see Section 4 for the full list of available cities
DATASET = "a4i_climate"   # the BigQuery "folder" everything goes into
LOCATION = "US"           # keep every table in the same location
GHCN_FIRST_YEAR = 2019    # how far back to pull daily station history
GHCN_LAST_YEAR = 2025

# --- Timing, so you know where your minutes went --------------------------
import time
NOTEBOOK_START = time.time()
STEP_TIMES = {}

class step:
    '''Context manager that times a block and reports it.'''
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        elapsed = time.time() - self.t0
        STEP_TIMES[self.name] = elapsed
        print(f"\n[{self.name}] took {elapsed:.1f}s")
        return False

# --- Work out which project we are in ----------------------------------------
import os
import google.auth

credentials, PROJECT_ID = google.auth.default()

# Colab Enterprise usually supplies this; fall back to the environment if not.
if not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")

if not PROJECT_ID:
    raise RuntimeError(
        "Could not determine your project ID. Set it manually:\n"
        "    PROJECT_ID = 'your-project-id-here'"
    )

print(f"Project:  {PROJECT_ID}")
print(f"Dataset:  {DATASET}")
print(f"Location: {LOCATION}")
print(f"City:     {CITY}")

Now we create the dataset. This is the one piece of setup that writes anything.

`exists_ok=True` means you can run this cell as many times as you like without it failing—which
matters, because you *will* re-run cells today.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
dataset_ref.location = LOCATION
dataset = client.create_dataset(dataset_ref, exists_ok=True)

print(f"Ready: {dataset.full_dataset_id}  (location: {dataset.location})")

---

# 2. Start with the wrong answer

## Why we are doing this on purpose

Before we show you the good data, we want you to see the obvious approach fail. It will save
you an hour later, and it will tell you something true about the problem.

The obvious approach is national weather data. NOAA publishes daily observations from weather
stations going back to 1929, it is already in BigQuery, and it is free to query. If someone
asked you to predict extreme heat, this is where you would start.

Let's ask it a microclimate question and watch what happens.

**The dataset:** `bigquery-public-data.ghcn_d` is NOAA's Global Historical Climatology Network,
Daily. One row per station, per day, per measurement. We will come back to it properly in
Section 7, because it is genuinely useful—just not for this.

The query below finds weather stations inside Chicago and asks each one how hot it got on a
specific summer day.

In [ ]:
%%bigquery --project $PROJECT_ID

-- How hot was it in Chicago on 2021-08-25, according to the weather stations?
--
-- GHCN stores temperature in TENTHS of a degree Celsius, so we divide by 10
-- and convert to Fahrenheit. More on that in Section 7.

SELECT
  s.name AS station_name,
  ROUND(s.latitude, 3)  AS lat,
  ROUND(s.longitude, 3) AS lon,
  ROUND((o.value / 10.0) * 9/5 + 32, 1) AS max_temp_f
FROM `bigquery-public-data.ghcn_d.ghcnd_2021` AS o
JOIN `bigquery-public-data.ghcn_d.ghcnd_stations` AS s
  ON o.id = s.id
WHERE o.element = 'TMAX'
  AND o.date = DATE '2021-08-25'
  AND o.qflag IS NULL                     -- exclude observations that failed NOAA's QA
  AND s.latitude  BETWEEN 41.64 AND 42.03  -- roughly the city of Chicago
  AND s.longitude BETWEEN -87.94 AND -87.52
ORDER BY max_temp_f DESC

## Look at what came back

A handful of rows, and every one of them says roughly the same number.

That is not a bug. It is exactly what this data is for. Weather stations are sparse and mostly
sited at airports, and they are measuring the regional air mass—the weather *over* the city.
For "will there be a heat wave on Thursday," this is the right tool and it is excellent.

But our planner is not asking that. Our planner is asking **which corner**. And the honest
answer from this data is: *the whole city is the same temperature.* Which we know is false.

Here is the trap you have just avoided. You could train a model on this and it would score
beautifully—high accuracy, tight error bars, a chart that looks great on a slide. It would have
learned that summer is hot. Every table that reaches for the obvious dataset builds that model,
and it does not survive the first question from a judge.

**What we actually need is a thermometer that moved around the city.** That exists.

---

# 3. NOAA Heat Watch: a thermometer that moved

## What it is

Starting in 2017, NOAA and CAPA Strategies began running **community heat mapping campaigns**.
Here is how one works.

A city picks its hottest likely day. Volunteers—ordinary residents, not scientists—mount
calibrated sensors on their cars and drive assigned routes covering the whole city. They go out
three times: morning, mid-afternoon, and evening. The sensors log air temperature, heat index,
and relative humidity roughly **once per second**.

The result is tens of thousands of temperature readings scattered along the actual streets of
one city, all collected within the same one-hour window.

## Why this changes the problem

Because every reading is taken at nearly the same time, the differences between them are not
weather. They are **the city itself**: pavement, buildings, tree cover, water, traffic. That is
the microclimate signal, and it is the only dataset in this challenge that contains it.

Typical campaign: **10,000 to 30,000 points** for one city.

## What it cannot do, and you must say this out loud

Heat Watch is **one day**. It is not a forecast and it is not a climate record. It tells you the
*spatial pattern* of heat in a city—which is remarkably stable from year to year, because
buildings and roads do not move—but it cannot tell you *when* the next hot day arrives.

That is why we pair it with GHCN in Section 7. **Heat Watch gives you where. GHCN gives you
when.** Neither is sufficient alone, and a judge will absolutely ask you which one you used for
which. Know the answer.

## Licensing, because you should always check

NOAA publishes this under the **Creative Commons Zero 1.0 Universal Public Domain Dedication**:

> "These data were produced by NOAA and are not subject to copyright protection in the United
> States. NOAA waives any potential copyright and related rights in these data worldwide through
> the Creative Commons Zero 1.0 Universal Public Domain Dedication (CC0 1.0)"

CC0 is about as permissive as it gets: commercial use is fine, attribution is not required, and
there is no share-alike obligation on anything you build. There is no personal data—these are
sensor readings from vehicles on public streets.

We still recommend crediting NOAA and CAPA Strategies in your demo. It costs you one line and
volunteers drove those routes.

---

# 4. Finding your city

## The problem

The data lives in an ArcGIS REST service, which is a mapping server rather than a database. It
organises data into numbered **layers**, one per city campaign.

You would reasonably expect layer names to tell you which city each one is. They do not. Most of
the 88 layers are named `af_trav`. A few have real names. The only reliable way to identify a
city is to look at each layer's **bounding box**—the rectangle of latitude and longitude it
covers—and work out what is inside it.

We did that for you. Every layer below was identified by fetching its extent and matching the
centre point to a city. That is the sort of unglamorous work that eats a hackathon morning, so
we did it in advance.

## Which cities we have verified

We tested the loading and cleaning code below against the cities in `VERIFIED_CITIES`. Those are
the ones we are confident about.

The other campaigns in `MORE_CITIES` should work—the code is written to adapt—but we have not
run every one. **If you pick one and something looks wrong, fall back to a verified city rather
than spending your morning debugging our loader.** You have four and a half hours.

One absence worth knowing about: **Phoenix is not in this dataset.** Neither are San Antonio,
Denver, Minneapolis, Charlotte, Indianapolis, or St. Louis. Coverage follows where communities
organised volunteer campaigns, not city size. If someone asks you why you did not use the
hottest big city in America, that is the answer.

In [ ]:
# Afternoon campaign layers, identified by bounding box.
# Afternoon is the hottest window and the one this notebook uses.

VERIFIED_CITIES = {
    "New York (Manhattan/Bronx)": 7,
    "New York (Brooklyn)":       67,
    "Los Angeles (Central)":     46,
    "Los Angeles (San Fernando)": 47,
    "Chicago":                   75,
    "Houston":                   90,
    "Philadelphia":              61,
    "San Diego":                  0,
    "Dallas":                    86,
    "San Jose":                  26,
    "Austin":                    37,
    "Jacksonville":              58,
    "San Francisco":             66,
    "Columbus":                  59,
    "Seattle":                   25,
    "Washington DC":             53,
    "Boston":                    52,
    "El Paso":                   35,
    "Nashville":                 68,
    "Detroit":                   31,
    "Portland OR":               88,
    "Las Vegas":                 65,
    "Oklahoma City":             81,
    "Milwaukee":                 55,
    "Albuquerque":               22,
    "Baltimore":                 54,
    "Sacramento":                42,
    "Atlanta":                    2,
    "Kansas City MO":            10,
    "Miami":                     32,
    "Oakland/Berkeley":          44,
    "New Orleans":               33,
    "Honolulu":                  50,
    "Vancouver BC":              36,
}

MORE_CITIES = {
    "Raleigh-Durham": 6,  "Charleston SC": 4,   "Richmond VA": 15,
    "Newark/Jersey City": 11, "Virginia Beach": 13, "Cincinnati": 29,
    "Providence": 28,     "Spokane": 56,        "Omaha": 57,
    "Boulder": 60,        "Knoxville": 63,      "Columbia MO": 64,
    "Columbia SC": 72,    "Salt Lake City": 79, "Little Rock": 82,
    "Asheville": 83,      "Wilmington DE": 87,  "Toledo": 89,
    "Scranton/Wilkes-Barre": 78, "Cedar Rapids": 80, "Jackson MS": 30,
    "Fort Lauderdale": 51, "West Palm Beach": 39, "Worcester": 38,
    "Yonkers": 45,        "Burlington VT": 24,  "Las Cruces": 23,
    "Boise": 49,          "Nampa": 48,          "Sedona": 73,
    "San Bernardino": 41, "Victorville": 40,    "Longmont": 77,
    "Kansas City KS": 76, "San Juan PR": 43,
}

ALL_CITIES = {**VERIFIED_CITIES, **MORE_CITIES}

if CITY not in ALL_CITIES:
    raise ValueError(
        f"'{CITY}' is not a Heat Watch campaign city.\n\n"
        f"Verified cities: {', '.join(sorted(VERIFIED_CITIES))}"
    )

LAYER_ID = ALL_CITIES[CITY]
tier = "verified" if CITY in VERIFIED_CITIES else "untested (fall back if it misbehaves)"
print(f"{CITY}: afternoon layer {LAYER_ID}  [{tier}]")

---

# 5. Pulling the raw data

## Read the API's limits before you trust it

This is a habit worth forming. Before pulling from any service, find out what it will refuse to
do. This one has four quirks, and each would cost you time if you met it by surprise:

1. **It caps responses at 100,000 records.**
2. **Pagination is switched off.** The usual `resultRecordCount` parameter is rejected outright
   with `"Pagination is not supported."` So the normal way of asking for data in pages does not
   work at all.
3. **Houston has 146,963 points**, which is more than the cap. With pagination unavailable, the
   only way to get all of it is to slice on the row ID—`WHERE FID >= 0 AND FID < 25000`, then the
   next slice, and so on.
4. **Asking for rows past the end of the data times out.** Not an empty result—a **504 Gateway
   Time-out**, after the better part of a minute. So the loop below stops as soon as it has the
   number of rows the service told us to expect, rather than probing until it runs out.

That last one is worth sitting with, because it is a real lesson about other people's servers.
The obvious way to write a chunked loop is "keep asking until you get nothing back." Here, that
costs you a timeout every single run. Ask how many rows exist first, then take exactly that many.

## Two things we do to be a good client

**We retry.** This server returns 5xx errors under load, and on event day a room full of people
will be hitting it at once. One failure is not a reason to stop—we wait a moment and try again,
backing off each time. Code that talks to a network without a retry is code that works on your
laptop and fails in the room.

**We ask only for the columns we need.** Some campaigns publish thirty fields of raw GPS logger
output—satellite counts, battery voltage, course heading. We want six. Requesting only those
cuts the response size substantially, and it is free to do. We still *print* the full field list
below, because you should see what a campaign actually publishes.

One last oddity: **asking for a layer that does not exist returns HTTP 500, not 404.** If you see
a persistent 500, you probably have a bad layer number rather than a broken server.

In [ ]:
import requests
import pandas as pd
import time as _time

BASE = "https://gis.nnvl.noaa.gov/arcgis/rest/services/HINDZ"
SERVICE = f"{BASE}/Afternoon_Traverses_in_Cities/MapServer"
CHUNK = 25_000     # comfortably under the 100k cap
RETRIES = 3

# The only fields we can actually use. Everything else in a layer is weight on the wire.
# Some campaigns ship 30 columns of raw GPS logger output; we want six of them.
WANTED = {
    "temp_f", "t_f", "T_F", "temp_c", "t_c", "T_C", "T", "Temprtr",
    "hi_f", "HI_F", "ht_ndx_f", "hi_c", "HI_C", "ht_ndx_c",
    "rh", "RH", "Humidty",
    "datetime", "dttm_lc", "dttm_tc",
}


def _get(url, params, what):
    '''GET with retry. This server returns 5xx under load, so one failure is not fatal.'''
    last_error = None
    for attempt in range(RETRIES):
        try:
            r = requests.get(url, params=params, timeout=180)
            if r.status_code in (500, 502, 503, 504):
                last_error = f"HTTP {r.status_code}"
                wait = 2 ** attempt
                print(f"    [{what}] {last_error}, retrying in {wait}s")
                _time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except requests.exceptions.RequestException as exc:
            last_error = f"{type(exc).__name__}"
            wait = 2 ** attempt
            print(f"    [{what}] {last_error}, retrying in {wait}s")
            _time.sleep(wait)
    raise RuntimeError(f"{what} failed after {RETRIES} attempts ({last_error})")


def layer_fields(layer_id: int):
    '''Every field name this layer publishes.'''
    meta = _get(f"{SERVICE}/{layer_id}", {"f": "json"}, "layer metadata")
    if "error" in meta:
        raise RuntimeError(f"Layer {layer_id} not found: {meta['error']}")
    return [f["name"] for f in meta.get("fields", [])]


def layer_count(layer_id: int) -> int:
    '''How many points are in this city's campaign?'''
    payload = _get(
        f"{SERVICE}/{layer_id}/query",
        {"where": "1=1", "returnCountOnly": "true", "f": "json"},
        "record count",
    )
    if "error" in payload:
        raise RuntimeError(f"Service error for layer {layer_id}: {payload['error']}")
    return payload["count"]


def fetch_layer(layer_id: int) -> pd.DataFrame:
    '''Pull every point from a layer, slicing on the row ID to stay under the cap.'''
    total = layer_count(layer_id)
    all_fields = layer_fields(layer_id)

    oid = "FID" if "FID" in all_fields else all_fields[0]
    keep = [f for f in all_fields if f in WANTED]
    out_fields = ",".join([oid] + keep) if keep else "*"

    print(f"Layer {layer_id}: {total:,} points")
    print(f"Fields published : {all_fields}")
    print(f"Fields requested : {[oid] + keep}   ({len(keep) + 1} of {len(all_fields)})")
    print()

    rows = []
    lo = 0
    ceiling = total + CHUNK * 4   # guard against sparse row IDs looping forever

    while len(rows) < total and lo < ceiling:
        payload = _get(
            f"{SERVICE}/{layer_id}/query",
            {
                "where": f"{oid} >= {lo} AND {oid} < {lo + CHUNK}",
                "outFields": out_fields,
                "returnGeometry": "true",
                "f": "json",
            },
            f"rows {lo:,}-{lo + CHUNK:,}",
        )
        features = payload.get("features", [])
        for feat in features:
            record = dict(feat["attributes"])
            geom = feat.get("geometry") or {}
            record["_x"] = geom.get("x")
            record["_y"] = geom.get("y")
            rows.append(record)

        if features:
            print(f"  ... {len(rows):,} of {total:,}")
        lo += CHUNK

    return pd.DataFrame(rows)


with step("fetch heat watch"):
    raw = fetch_layer(LAYER_ID)

print(f"\nRetrieved {len(raw):,} rows.")
raw.head()

---

# 6. Cleaning: three real problems

Look at the column names printed above, then look at this. These are all real column sets from
different cities in **the same service**:

| City | Temperature column | Humidity | Time |
|---|---|---|---|
| San Diego | `temp_f` | `rh` | `datetime` |
| Houston | `t_f` | `rh` | *(none)* |
| Boston | `T_F` and `T_C` | `RH` | `time_n` (an integer like `192147`) |
| Washington DC | `T` (Celsius, unlabelled) | *(none)* | `loctime` (time only) |
| San Jose | `Temprtr` (Celsius) | `rh` | `dttm_lc` plus a `tz` column |

Each campaign was processed by whoever ran it, and nobody harmonised the results afterwards.
This is completely normal for public data and it is worth seeing rather than being shielded
from. Three problems, in increasing order of how badly they will hurt you.

### Problem 1: the temperature column has seven different names

`temp_f`, `t_f`, `T_F`, `Temprtr`, `t_c`, `temp_c`, `T`. We handle this by looking for any of
them rather than hard-coding one per city. The alternative is 88 special cases, which is not
code anyone should maintain.

### Problem 2: some cities are in Celsius, and they do not say so

Washington DC's temperature column is called `T` and the values look like `35.7`. If you assume
Fahrenheit you have just recorded a mild spring afternoon in what was actually a dangerous heat
event. We detect the unit from the column name and convert.

### Problem 3: missing values are recorded as −9999, and this is the one that will silently ruin your model

Milwaukee returns `-9999.0` for humidity and heat index where the sensor did not report. It is
not null. It is not blank. It is a number, and every tool you use will treat it as one.

Feed that into a regression and your model will learn that some places are ten thousand degrees
below zero. It will not crash. It will not warn you. It will just be wrong, and the error will
be large enough to distort every other coefficient.

**This is the single most common way a hackathon model quietly fails.** Sentinel values are
everywhere in public data—GHCN uses `9999.9` for the same purpose. Always look at the minimum
value of a numeric column before you trust it.

There is a fourth, smaller one: Sedona reports humidity as a fraction between 0 and 1 while
everyone else uses a percentage. We detect and rescale it.

In [ ]:
import numpy as np

# Candidate column names, gathered by inspecting every layer in the service.
TEMP_F_COLS = ["temp_f", "t_f", "T_F"]
TEMP_C_COLS = ["temp_c", "t_c", "T_C", "T", "Temprtr"]
HEAT_F_COLS = ["hi_f", "HI_F", "ht_ndx_f"]
HEAT_C_COLS = ["hi_c", "HI_C", "ht_ndx_c"]
RH_COLS     = ["rh", "RH", "Humidty"]
TIME_COLS   = ["datetime", "dttm_lc", "dttm_tc"]

SENTINELS = [-9999.0, -999.0, 9999.9]


def first_present(df, candidates):
    '''Return the first candidate column that exists in df, else None.'''
    for name in candidates:
        if name in df.columns:
            return name
    return None


def c_to_f(series):
    return series * 9.0 / 5.0 + 32.0


def normalize(df, city):
    '''Turn any campaign's column set into one predictable schema.'''
    out = pd.DataFrame()
    out["city"] = city
    out["longitude"] = df["_x"]
    out["latitude"] = df["_y"]

    # --- Temperature: prefer a Fahrenheit column, else convert from Celsius ---
    tf, tc = first_present(df, TEMP_F_COLS), first_present(df, TEMP_C_COLS)
    if tf:
        out["temp_f"] = pd.to_numeric(df[tf], errors="coerce")
        temp_source = f"{tf} (already Fahrenheit)"
    elif tc:
        out["temp_f"] = c_to_f(pd.to_numeric(df[tc], errors="coerce"))
        temp_source = f"{tc} (converted from Celsius)"
    else:
        raise RuntimeError(f"No temperature column found. Columns: {list(df.columns)}")

    # --- Heat index: same treatment, but it is allowed to be missing ---
    hf, hc = first_present(df, HEAT_F_COLS), first_present(df, HEAT_C_COLS)
    if hf:
        out["heat_index_f"] = pd.to_numeric(df[hf], errors="coerce")
    elif hc:
        out["heat_index_f"] = c_to_f(pd.to_numeric(df[hc], errors="coerce"))
    else:
        out["heat_index_f"] = np.nan

    # --- Relative humidity ---
    rh = first_present(df, RH_COLS)
    out["humidity_pct"] = pd.to_numeric(df[rh], errors="coerce") if rh else np.nan

    # --- Timestamp, where the campaign recorded one ---
    tcol = first_present(df, TIME_COLS)
    out["observed_at"] = pd.to_datetime(df[tcol], errors="coerce") if tcol else pd.NaT

    # --- Sentinel removal. Do this BEFORE any range check. ---
    numeric = ["temp_f", "heat_index_f", "humidity_pct"]
    sentinel_hits = 0
    for col in numeric:
        for bad in SENTINELS:
            hits = (out[col] == bad).sum()
            if hits:
                sentinel_hits += int(hits)
                out.loc[out[col] == bad, col] = np.nan
        # A Celsius sentinel becomes a huge negative after conversion; catch those too.
        out.loc[out[col] < -200, col] = np.nan

    # --- Humidity recorded as a 0-1 fraction rather than a percentage ---
    if out["humidity_pct"].notna().any() and out["humidity_pct"].max() <= 1.5:
        out["humidity_pct"] = out["humidity_pct"] * 100.0
        rh_note = "rescaled from 0-1 fraction to percent"
    else:
        rh_note = "already a percentage"

    # --- Drop rows with no usable temperature or position ---
    before = len(out)
    out = out.dropna(subset=["temp_f", "latitude", "longitude"]).reset_index(drop=True)

    print(f"Temperature source : {temp_source}")
    print(f"Humidity           : {rh_note}")
    print(f"Sentinel values replaced with null: {sentinel_hits:,}")
    print(f"Rows dropped for missing temp/position: {before - len(out):,}")
    print(f"Clean rows: {len(out):,}")
    return out


clean = normalize(raw, CITY)
clean.head()

## Check your work before you trust it

Two lines of defence, and you should run both every time you clean anything.

**Look at the range.** Air temperature on a campaign day should sit somewhere between about 55°F
and 120°F. If your minimum is negative, a sentinel got through. If your maximum is 300, a unit
conversion is wrong.

**Look at the spread.** This is also the first genuinely interesting result of the day.

In [ ]:
summary = clean[["temp_f", "heat_index_f", "humidity_pct"]].describe().T
print(summary[["count", "mean", "min", "max"]].round(1).to_string())

lo, hi = clean["temp_f"].min(), clean["temp_f"].max()
print(f"\n{CITY}: coolest reading {lo:.1f}°F, hottest {hi:.1f}°F")
print(f"Spread across the city within the same hour: {hi - lo:.1f}°F")

if lo < 30 or hi > 140:
    print("\n!! Range looks wrong. A sentinel or a unit conversion has slipped through.")

## That number is your challenge

Look at the spread you just printed. In most campaign cities it lands somewhere between ten and
twenty degrees Fahrenheit—**measured at the same hour, on the same day, in the same city.**

Compare that with Section 2, where every weather station in Chicago reported essentially one
temperature.

That gap is the entire reason this challenge exists. A planner choosing between two
intersections is choosing between two genuinely different environments, and until now nothing in
their data told them so. You are about to build the thing that does.

---

# 7. Loading into BigQuery

Everything so far has been in memory in this notebook. If the runtime restarts, it is gone—and
more importantly, your teammates cannot reach it and BigQuery ML cannot train on it.

So we write it to a table. From here on, this data is a permanent thing in your project that
anyone on your team can query.

In [ ]:
def load_table(df, table_name, description=""):
    '''Write a dataframe into BigQuery, replacing whatever was there.'''
    table_id = f"{PROJECT_ID}.{DATASET}.{table_name}"
    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
    )
    job.result()

    table = client.get_table(table_id)
    if description:
        table.description = description
        client.update_table(table, ["description"])

    print(f"{table_id}: {table.num_rows:,} rows, {len(table.schema)} columns")
    return table_id


with step("load heat_observations"):
    heat_table = load_table(
        clean,
        "heat_observations",
        "NOAA/CAPA Heat Watch afternoon traverse observations. CC0 1.0. "
        "Single-day campaign snapshot: spatial pattern only, not a climatology.",
    )

Let's ask BigQuery a question we could not have asked before: **where in this city is it
hottest, and how much hotter is that than the city's own average?**

This is a small preview of what your model has to learn to predict.

In [ ]:
%%bigquery --project $PROJECT_ID

WITH city_mean AS (
  SELECT AVG(temp_f) AS avg_temp_f
  FROM `{PROJECT}.{DATASET}.heat_observations`
)
SELECT
  ROUND(latitude, 4)  AS lat,
  ROUND(longitude, 4) AS lon,
  ROUND(temp_f, 1)    AS temp_f,
  ROUND(temp_f - (SELECT avg_temp_f FROM city_mean), 1) AS degrees_above_city_average
FROM `{PROJECT}.{DATASET}.heat_observations`
ORDER BY temp_f DESC
LIMIT 15

> **Note on the cell above:** the `%%bigquery` magic does not substitute Python variables inside
> the SQL. If it errors, replace `{PROJECT}.{DATASET}` with your actual project and dataset
> names, or use the `client.query(...)` form shown in later cells. We have left it in this shape
> because it reads clearly, and fixing it is a thirty-second exercise in seeing how the pieces
> connect.

That `degrees_above_city_average` column is worth staring at. It is the **offset**, and it is
what Section 9 will ask you to predict. Not the temperature—the *difference*. Hold that thought.

---

# 8. Adding time: NOAA GHCN-Daily

## Why we need it

Heat Watch is one day. Your planner will reasonably ask "how often does this happen?" and
"is it getting worse?"—and a single afternoon cannot answer either.

GHCN-Daily can. It holds daily observations from more than 100,000 stations worldwide, some
going back to the 1920s. It is the temporal backbone that Heat Watch lacks.

**The division of labour, which you should be able to state in one sentence at demo time:**
Heat Watch tells you *where within a city* it is hot. GHCN tells you *how often the city as a
whole* gets hot. Your model connects them.

## Three things about this dataset that will catch you

**It is stored long, not wide.** One row per station, per day, per *measurement type*. The
maximum temperature and the minimum temperature for the same station on the same day are two
separate rows, distinguished by the `element` column. If you are used to one row per
observation, this will feel inside out. Use conditional aggregation to pivot.

**Values are in tenths—but not all of them.** `TMAX`, `TMIN`, and `PRCP` are stored in tenths
(tenths of a degree Celsius, tenths of a millimetre). But `SNOW` and `SNWD` are in whole
millimetres. If you apply a blanket `value / 10` across every element, you have silently divided
all your snowfall by ten. We are only using TMAX here, but remember this if you extend to winter
hazards.

**Filter on the quality flag.** `qflag IS NULL` means the observation passed NOAA's quality
checks. Anything else means it failed one, and NOAA is telling you not to trust it.

In [ ]:
# Bounding box for the city, derived from the Heat Watch points, padded slightly
# so we catch nearby stations that represent the same regional air mass.
PAD = 0.25
lat_min, lat_max = clean["latitude"].min()  - PAD, clean["latitude"].max()  + PAD
lon_min, lon_max = clean["longitude"].min() - PAD, clean["longitude"].max() + PAD

print(f"Searching for stations in {lat_min:.2f}..{lat_max:.2f} N, {lon_min:.2f}..{lon_max:.2f} E")

# Which recent GHCN year tables actually exist? Do not assume.
years_df = client.query('''
    SELECT table_name
    FROM `bigquery-public-data.ghcn_d.INFORMATION_SCHEMA.TABLES`
    WHERE table_name LIKE 'ghcnd_2%'
    ORDER BY table_name DESC
    LIMIT 8
''').to_dataframe()
print("\nMost recent GHCN year tables available:")
print(years_df.to_string(index=False))

Now the daily history. We are pulling ten years of maximum temperature for stations near your
city, so you can characterise how often it crosses a heat threshold.

Note the wildcard table syntax: `ghcnd_20*` queries many yearly tables at once, and
`_TABLE_SUFFIX` lets us restrict which ones. This is how BigQuery handles datasets split into
one table per year.

In [ ]:
ghcn_sql = f'''
SELECT
  o.id AS station_id,
  s.name AS station_name,
  s.latitude,
  s.longitude,
  o.date AS observation_date,
  ROUND((o.value / 10.0) * 9/5 + 32, 1) AS max_temp_f
FROM `bigquery-public-data.ghcn_d.ghcnd_20*` AS o
JOIN `bigquery-public-data.ghcn_d.ghcnd_stations` AS s
  ON o.id = s.id
WHERE _TABLE_SUFFIX BETWEEN '{str(GHCN_FIRST_YEAR)[2:]}' AND '{str(GHCN_LAST_YEAR)[2:]}'
  AND o.element = 'TMAX'
  AND o.qflag IS NULL
  AND s.latitude  BETWEEN {lat_min} AND {lat_max}
  AND s.longitude BETWEEN {lon_min} AND {lon_max}
'''

ghcn_table = f"{PROJECT_ID}.{DATASET}.station_daily_temps"

with step("ghcn query"):
    job = client.query(
        ghcn_sql,
        job_config=bigquery.QueryJobConfig(
            destination=ghcn_table, write_disposition="WRITE_TRUNCATE"
        ),
    )
    job.result()

table = client.get_table(ghcn_table)
print(f"{ghcn_table}: {table.num_rows:,} rows")
print(f"Bytes processed: {job.total_bytes_processed / 1e9:.2f} GB")

That `Bytes processed` number is worth noticing. BigQuery charges by data scanned, and the free
tier covers 1 TiB per month. Nothing in this notebook comes close—but the habit of glancing at
that number will serve you well when someone suggests `SELECT *` on a hundred-year table.

---

# 9. Who lives there, and one decision we made deliberately

## Why demographics matter here

A hot block with nobody on it is a curiosity. A hot block where elderly residents live without
air conditioning or a car is an emergency. Your planner is siting a limited number of cooling
centres, and *who is exposed* is at least as important as *how hot it is*.

The American Community Survey gives us that at census tract level: population, age, income,
vehicle access, housing age. All public domain, all aggregate—there is no individual-level data
in it, by law.

## A decision we made for you, and why

**We are not loading the race and ethnicity columns.** ACS publishes them—`black_pop`,
`hispanic_pop`, `white_pop`, `asian_pop`—and we are deliberately leaving them out. Here is the
reasoning, because you may be asked about it and because the reasoning is more interesting than
the rule.

**Race correlates strongly with urban heat exposure in American cities.** This is one of the
better-established findings in the field: neighbourhoods that were redlined in the 1930s run
measurably hotter today than neighbourhoods that were not, often by five to seven degrees. If
you put race into a model of heat exposure, you will find real signal. We are not pretending
otherwise.

**But the correlation is not the cause.** Race does not make asphalt absorb sunlight. It
correlates with heat because it correlates with the physical characteristics of a
place—pavement, building density, street layout, tree cover—which are themselves the product of
decades of investment decisions. The heat comes from the concrete. The race of the people living
next to the concrete is a downstream fact.

**And those physical characteristics are already in our data.** That is the practical argument.
Once your model has street density, land-use mix, and housing age, race adds very little that is
not already accounted for. It is a cruder measurement of something we can measure directly.

So we drop it—not because it predicts nothing, but because it is the *worse* variable, and
because a model that allocates public resources on the basis of race is something you do not
want to have built, whatever its R².

## The part that matters most

**Removing the column does not make your model unbiased.** This is the trap, and it catches
experienced people.

Every proxy is still sitting in your feature set. Street density, housing age, and land use all
remain correlated with race, so a model trained without the race column can still produce
recommendations that fall along racial lines. Deleting a column proves nothing about your
output. It is sometimes called *fairness through unawareness*, and it does not work.

The thing that actually works is checking. When you have your ranked list of recommended
locations in Section 10, go back and look at who those neighbourhoods are—as an **audit of your
results**, never as an input to the model. If your recommendations systematically miss the most
vulnerable areas, that is a finding about your model, and it is worth more in a demo than
another percentage point of accuracy.

**Strip the variable. Audit the outcome. Say you did both.**

One note if you go looking for other vulnerability data: the NYC Heat Vulnerability Index and
the CDC Social Vulnerability Index both bake race into a single precomputed score. You cannot
remove it, because there is no column to drop—it is inside the number. If you use CDC's index,
use Themes 1, 2 and 4 individually rather than the combined `RPL_THEMES` value.

In [ ]:
# Census tracts intersecting the city, with the demographics that bear on heat
# vulnerability. Race and ethnicity columns are deliberately not selected.

acs_sql = f'''
SELECT
  acs.geo_id,
  geo.internal_point_lat  AS tract_lat,
  geo.internal_point_lon  AS tract_lon,
  acs.total_pop,
  acs.median_age,
  acs.median_income,
  acs.housing_units,
  acs.median_year_structure_built,
  acs.no_cars,
  acs.households,
  SAFE_DIVIDE(acs.poverty, acs.pop_determined_poverty_status) AS poverty_rate,
  SAFE_DIVIDE(acs.no_cars, acs.households)                    AS no_vehicle_rate
FROM `bigquery-public-data.census_bureau_acs.censustract_2021_5yr` AS acs
JOIN `bigquery-public-data.geo_census_tracts.us_census_tracts_national` AS geo
  ON acs.geo_id = geo.geo_id
WHERE geo.internal_point_lat BETWEEN {lat_min} AND {lat_max}
  AND geo.internal_point_lon BETWEEN {lon_min} AND {lon_max}
'''

acs_table = f"{PROJECT_ID}.{DATASET}.tract_demographics"

with step("acs query"):
    job = client.query(
        acs_sql,
        job_config=bigquery.QueryJobConfig(
            destination=acs_table, write_disposition="WRITE_TRUNCATE"
        ),
    )
    job.result()

table = client.get_table(acs_table)
print(f"{acs_table}: {table.num_rows:,} census tracts")

Two ACS quirks worth knowing, both visible in the query above.

**There is no poverty rate column.** ACS gives you a count (`poverty`) and its denominator
(`pop_determined_poverty_status`) and expects you to divide. We use `SAFE_DIVIDE` so that a tract
with zero population returns null rather than crashing the query.

**There is no `pop_over_65` column either.** If you want the elderly population—and for heat
vulnerability you probably do—you have to sum twelve separate columns, `male_65_to_66` through
`female_85_and_over`. We have left that as an exercise because it is a good one: it is exactly
the kind of small, annoying, real data-engineering task that separates a working prototype from
a demo.

---

# 10. Framing your model

This is where the notebook stops and your team starts. Your required differentiating technology
is **BigQuery ML**, and the model is yours to design. What follows is the problem statement and
the constraints—not the answer.

## Predict the offset, not the temperature

The instinct is to predict temperature. Resist it. A model that predicts temperature will spend
all its effort learning that afternoons are warmer than mornings and summer is warmer than
spring, which everyone already knows.

Predict the **offset** instead: *how many degrees hotter is this location than the regional
station reading at the same time?* You saw this column in Section 7.

That reframing does three things. It removes the seasonal signal, so the model has to learn the
city rather than the calendar. It makes the output directly useful, because a planner can add
your offset to any forecast. And it makes the explanation meaningful, which matters for the next
point.

## What the free tier covers

BigQuery ML has two families of model, and the distinction affects your budget:

| Trains inside BigQuery — free tier applies | Trains externally on Vertex AI — **not** free-tier covered |
|---|---|
| `LINEAR_REG` | `BOOSTED_TREE_REGRESSOR` |
| `LOGISTIC_REG` | `RANDOM_FOREST_REGRESSOR` |
| `KMEANS` | `DNN_REGRESSOR` |
| `PCA` | `AUTOML_REGRESSOR` |
| `ARIMA_PLUS` | |

The right-hand column will produce a bill. It may still be the right choice for your team, but
make it a decision rather than a surprise.

**We suggest `LINEAR_REG` on the offset.** It trains in minutes, it is free-tier eligible, and—
most importantly—it supports the function below.

## `ML.EXPLAIN_PREDICT` is the whole challenge

Re-read the challenge statement. It does not ask for a location. It asks for a location **with
the reasoning behind it**.

`ML.EXPLAIN_PREDICT` returns per-feature attributions alongside each prediction: not just
"+6.2°F" but "*+4.1 from street density, +2.8 from building age, −0.7 from park proximity.*"

That is not a nice extra. That sentence **is** your Placement Recommendation. It is what a
planner puts in front of a city council when someone asks why this intersection and not that
one. Teams that treat explanation as a reporting afterthought end up with a number and no
argument.

## Features available to you

From the tables you just built: latitude and longitude, time of day, tract population density,
median income, housing age, vehicle access, poverty rate.

**Worth adding if you have time:** EPA's Smart Location Database
(`https://edg.epa.gov/EPADataCommons/public/OA/WalkabilityIndex.zip`) gives block-group street
intersection density, land-use mix, and transit access—`NatWalkInd` is the composite index. It
is CC0 and it is the strongest built-environment signal available nationally. One warning:
`D4A_Ranked` has a discontinuous scale where 1 means no transit access and the rest run 13–20,
with nothing in between. It is not a linear feature.

**Where strong teams will separate:** tree canopy. There is no clean national canopy layer that
loads in a hackathon window, but several cities publish their own—New York's street tree census
is even in BigQuery already, at `bigquery-public-data.new_york_trees.tree_census_2015`. If your
city has one, canopy is likely to be the single most explanatory feature you can add, and it
will show up beautifully in your attributions.

## Your output artifact

A **Placement Recommendation**: a ranked list of locations with coordinates, the predicted heat
offset, who lives nearby, and a drafted proposal a planner could actually take to a council
meeting.

Then the audit from Section 9. Look at where your recommendations landed and who lives there.
Report it whether or not it flatters you.

---

## Hand off to your team

Your agent lane needs to know these table names. Send them this list.

In [ ]:
print(f"Project: {PROJECT_ID}")
print(f"Dataset: {DATASET}\n")
for t in client.list_tables(f"{PROJECT_ID}.{DATASET}"):
    tbl = client.get_table(t)
    print(f"  {tbl.table_id:<22} {tbl.num_rows:>10,} rows")
print("\nYour agent's custom tool will query these. Your BQML model trains on them.")
print("Nothing else in this notebook is done for you, and that is the point. Go build.")

---

# Appendix: diagnostic summary

Run this if something looks wrong and you need to ask for help. It prints one block you can
paste into your team chat or hand to a coach—it is far more useful than a screenshot.

In [ ]:
print("=" * 64)
print("A4I CHALLENGE 1 - DIAGNOSTIC SUMMARY")
print("=" * 64)
print(f"Project        : {PROJECT_ID}")
print(f"Dataset        : {DATASET} ({LOCATION})")
print(f"City           : {CITY}  (afternoon layer {LAYER_ID})")
print(f"Verified city  : {CITY in VERIFIED_CITIES}")
print("-" * 64)

try:
    print(f"Raw rows fetched   : {len(raw):,}")
    print(f"Raw columns        : {sorted(raw.columns)}")
    print(f"Clean rows         : {len(clean):,}")
    print(f"Temp range (F)     : {clean['temp_f'].min():.1f} to {clean['temp_f'].max():.1f}")
    print(f"Spread (F)         : {clean['temp_f'].max() - clean['temp_f'].min():.1f}")
    print(f"Heat index nulls   : {clean['heat_index_f'].isna().sum():,}")
    print(f"Humidity range     : {clean['humidity_pct'].min():.1f} to {clean['humidity_pct'].max():.1f}")
    print(f"Timestamps present : {clean['observed_at'].notna().sum():,} of {len(clean):,}")
except Exception as exc:
    print(f"Cleaning stage did not complete: {exc}")

print("-" * 64)
try:
    for t in client.list_tables(f"{PROJECT_ID}.{DATASET}"):
        tbl = client.get_table(t)
        cols = ", ".join(f.name for f in tbl.schema)
        print(f"{tbl.table_id}: {tbl.num_rows:,} rows")
        print(f"    {cols}")
except Exception as exc:
    print(f"Could not list tables: {exc}")

print("-" * 64)
print("TIMING")
for name, secs in STEP_TIMES.items():
    print(f"  {name:<26} {secs:>7.1f}s")
print(f"  {'TOTAL WALL CLOCK':<26} {time.time() - NOTEBOOK_START:>7.1f}s")
print("=" * 64)